# TravisTEC Audio CTC Training (Colab Version)

This notebook is designed to run on Google Colab to leverage high-end GPUs (like T4, V100, A100) for training the Speech-to-Text model.

**Instructions:**
1.  Open this notebook in Google Colab.
2.  Change the Runtime Type to **GPU** (Runtime > Change runtime type > T4 GPU).
3.  Upload your `datasets` folder or clone your repository.
4.  Run the cells below.

In [ ]:
# 1. Check GPU Availability
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

if len(tf.config.list_physical_devices('GPU')) == 0:
    print("WARNING: No GPU detected. Please go to Runtime > Change runtime type > Select GPU.")
else:
    print("GPU Detected! Ready for heavy training.")

In [ ]:
# 2. Install Dependencies
!pip install librosa pandas jiwer

In [ ]:
# 3. Setup Data (Option A: Clone Repo)
# Uncomment the lines below if you want to clone your repo directly
# !git clone https://github.com/jaredjimenez0022-oss/TravisTEC.git
# %cd TravisTEC/backend

# 3. Setup Data (Option B: Upload Zip)
# If you uploaded a zip file named 'datasets.zip', unzip it:
# !unzip datasets.zip -d .

In [ ]:
# 4. Imports and Configuration
import os
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Adjust these paths based on where your data is located in Colab
# If you cloned the repo and cd'd into backend, these might work as is.
# If you uploaded datasets folder to /content/datasets, adjust accordingly.
BASE_DIR = "/content" # Or current directory "."
DATASET_PATH = os.path.join(BASE_DIR, "datasets/audio")
WAVS_PATH = os.path.join(DATASET_PATH, "wavs")
METADATA_PATH = os.path.join(DATASET_PATH, "metadata.csv")
MODEL_SAVE_PATH = os.path.join(BASE_DIR, "audio_ctc_model.keras")

# Audio Parameters
SAMPLE_RATE = 22050
FFT_LENGTH = 384
FRAME_LENGTH = 256
FRAME_STEP = 160

# Training Parameters (HEAVY MODE for Colab)
BATCH_SIZE = 32 # Increased batch size for GPU
EPOCHS = 500

print(f"Dataset Path: {DATASET_PATH}")
print(f"Model Save Path: {MODEL_SAVE_PATH}")

In [ ]:
# 5. Data Loading Functions
def load_data():
    """Loads metadata and validates files exist."""
    if not os.path.exists(METADATA_PATH):
        print(f"Error: Metadata file not found at {METADATA_PATH}")
        return None
    
    df = pd.read_csv(METADATA_PATH)
    # Ensure files exist
    valid_data = []
    for idx, row in df.iterrows():
        wav_path = os.path.join(WAVS_PATH, str(row['filename']))
        if os.path.exists(wav_path):
            valid_data.append(row)
        else:
            print(f"Warning: File {wav_path} not found. Skipping.")
    
    if not valid_data:
        print("No valid audio files found.")
        return None
        
    return pd.DataFrame(valid_data)

def process_audio(file_path):
    """Reads wav file and converts to spectrogram."""
    # 1. Read wav
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE)
    
    # 2. Get Short-Time Fourier Transform (STFT)
    stft = librosa.stft(audio, n_fft=FFT_LENGTH, hop_length=FRAME_STEP, win_length=FRAME_LENGTH)
    
    # 3. Compute magnitude and log-scale
    spectrogram = np.abs(stft)
    spectrogram = np.log1p(spectrogram)
    
    # 4. Transpose to (Time, Features) for RNN
    spectrogram = np.transpose(spectrogram)
    
    # 5. Normalize
    means = np.mean(spectrogram, axis=0)
    stds = np.std(spectrogram, axis=0)
    spectrogram = (spectrogram - means) / (stds + 1e-10)
    
    return spectrogram

def create_dataset(df, char_to_num):
    """Creates a tf.data.Dataset generator."""
    
    def generator():
        for _, row in df.iterrows():
            # Process Audio
            wav_path = os.path.join(WAVS_PATH, str(row['filename']))
            try:
                spectrogram = process_audio(wav_path)
                # Process Label
                label = row['transcript'].lower()
                label = char_to_num(list(label))
                yield spectrogram, label
            except Exception as e:
                print(f"Error processing {wav_path}: {e}")

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(None, FFT_LENGTH // 2 + 1), dtype=tf.float32),
            tf.TensorSpec(shape=(None,), dtype=tf.int64)
        )
    )
    
    # Padding is crucial for variable length audio and text
    dataset = dataset.padded_batch(
        BATCH_SIZE,
        padded_shapes=(
            [None, FFT_LENGTH // 2 + 1], 
            [None]
        ),
        padding_values=(0.0, tf.constant(0, dtype=tf.int64)) # 0 for padding labels
    )
    
    return dataset

def CTCLoss(y_true, y_pred):
    """
    Compute the training-time loss value using CTC.
    """
    batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
    input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
    
    # Calculate label length by counting non-zero values (since 0 is padding)
    label_length = tf.reduce_sum(tf.cast(tf.not_equal(y_true, 0), tf.int64), axis=1)
    
    input_length = input_length * tf.ones(shape=(batch_len,), dtype="int64")

    loss = tf.nn.ctc_loss(
        labels=tf.cast(y_true, tf.int32),
        logits=y_pred,
        label_length=tf.cast(label_length, tf.int32),
        logit_length=tf.cast(input_length, tf.int32),
        logits_time_major=False,
        blank_index=-1
    )
    return tf.reduce_mean(loss)

In [ ]:
# 6. Model Definition (HEAVY ARCHITECTURE)
def build_model(input_dim, output_dim):
    """Builds the Deep Learning Model (CNN + RNN + CTC)."""
    
    input_spectrogram = layers.Input((None, input_dim), name="input")
    
    # CNN Layers
    x = layers.Conv1D(32, kernel_size=3, activation="relu", padding="same")(input_spectrogram)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    
    x = layers.Conv1D(64, kernel_size=3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    # Recurrent Layers (High Capacity for GPU)
    # 3 Layers of Bidirectional LSTMs with 256 units each
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(x)
    x = layers.Dropout(0.5)(x)
    
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(x)
    x = layers.Dropout(0.5)(x)
    
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(x)
    x = layers.Dropout(0.5)(x)

    # Output Layer
    x = layers.Dense(output_dim + 1, name="output")(x)

    model = keras.models.Model(inputs=input_spectrogram, outputs=x, name="Audio_CTC_Model_Heavy")
    return model

In [ ]:
# 7. Training Loop
print("--- Starting Audio CTC Training (Colab) ---")

# 1. Load Data
df = load_data()
if df is not None:
    print(f"Found {len(df)} audio samples.")
    
    # 2. Create Vocabulary
    all_text = "".join(df['transcript'].tolist()).lower()
    unique_chars = sorted(list(set(all_text)))
    print(f"Vocabulary: {unique_chars}")
    
    char_to_num = layers.StringLookup(vocabulary=unique_chars, mask_token='')
    num_to_char = layers.StringLookup(vocabulary=char_to_num.get_vocabulary(), mask_token='', invert=True)
    
    # 3. Prepare Dataset
    dataset = create_dataset(df, char_to_num)
    
    # 4. Build Model
    input_dim = FFT_LENGTH // 2 + 1
    vocab_size = char_to_num.vocabulary_size()
    
    model = build_model(input_dim, vocab_size)
    model.summary()
    
    # 5. Compile
    opt = keras.optimizers.Adam(learning_rate=5e-4)
    model.compile(optimizer=opt, loss=CTCLoss)
    
    # 6. Train
    print(f"Starting training for {EPOCHS} epochs...")
    
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=MODEL_SAVE_PATH,
            monitor="loss",
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="loss",
            factor=0.5,
            patience=20,
            min_lr=1e-6,
            verbose=1
        ),
        keras.callbacks.EarlyStopping(
            monitor="loss",
            patience=50,
            restore_best_weights=True,
            verbose=1
        )
    ]
    
    history = model.fit(dataset, epochs=EPOCHS, callbacks=callbacks)
    
    # 7. Save Final
    model.save(MODEL_SAVE_PATH)
    print(f"Model saved to {MODEL_SAVE_PATH}")
    
    # Save Vocab
    vocab_path = MODEL_SAVE_PATH.replace(".keras", "_vocab.txt")
    with open(vocab_path, "w") as f:
        f.write("".join(unique_chars))
    print(f"Vocabulary saved to {vocab_path}")

In [ ]:
# 8. Download Model
from google.colab import files

# Zip the model and vocab
!zip -r audio_ctc_model.zip audio_ctc_model.keras audio_ctc_model_vocab.txt

# Download
files.download('audio_ctc_model.zip')